In [0]:
# Databricks notebook source
# ================================
# NOTEBOOK 02 — DOUBLE JOIN
# Section 4.1 du papier :
# "creating Joint Table JT = {F, W_o, W_d, C}"
# ================================

# COMMAND ----------
# ================================
# CELLULE 1 — Charger FT et OT
# ================================
from pyspark.sql.functions import (
    col, expr, to_timestamp, lit,
    lpad, concat, floor
)

df_FT = spark.read.parquet(
    "/Volumes/workspace/default/outputs/FT/"
)
df_OT = spark.read.parquet(
    "/Volumes/workspace/default/outputs/OT/"
)

print(f"✅ FT chargée : {df_FT.count():,} vols")
print(f"✅ OT chargée : {df_OT.count():,} observations")
df_FT.show(3)
df_OT.show(3)

# COMMAND ----------
# ================================
# CELLULE 2 — PREMIER JOIN
# Section 4.1 du papier :
# W_o = météo origine de t_sd → t_sd-12h
# Clé join : (ORIGIN_AIRPORT_ID, DATE(DEP))
# ================================

# Préparer OT pour jointure origine
df_OT_origin = df_OT.select(
    col("AirportID").alias("ORIGIN_ID"),
    col("OBS_DATETIME").alias("OBS_DEP_DATETIME"),
    col("Temp").alias("orig_Temp"),
    col("Humidity").alias("orig_Humidity"),
    col("WindDirection").alias("orig_WindDir"),
    col("WindSpeed").alias("orig_WindSpeed"),
    col("Pressure").alias("orig_Pressure"),
    col("SkyCondition").alias("orig_SkyCondition"),
    col("Visibility").alias("orig_Visibility"),
    col("WeatherType").alias("orig_WeatherType")
)

# Premier Join :
# Pour chaque vol, trouver observations météo à l'origine
# dans la fenêtre [DEP_DATETIME - 12h, DEP_DATETIME]
# Le papier : "W_o = O(A_o, t_sd), O(A_o, t_sd-1h), ..., O(A_o, t_sd-12h)"

df_join1 = df_FT.join(
    df_OT_origin,
    (df_FT["ORIGIN_AIRPORT_ID"] == df_OT_origin["ORIGIN_ID"]) &
    (df_OT_origin["OBS_DEP_DATETIME"] >= 
     df_FT["DEP_DATETIME"] - expr("INTERVAL 12 HOURS")) &
    (df_OT_origin["OBS_DEP_DATETIME"] <= df_FT["DEP_DATETIME"]),
    "left"
)

print(f"✅ Après Premier Join : {df_join1.count():,} lignes")
df_join1.show(3, truncate=False)

# COMMAND ----------
# ================================
# CELLULE 3 — Garder observation
# météo la plus proche de t_sd
# Le papier : "we take the closest one
# to the weather observation time requested"
# ================================
from pyspark.sql.functions import (
    abs as spark_abs, row_number, unix_timestamp
)
from pyspark.sql.window import Window

# Calculer différence en secondes entre
# heure vol et heure observation
df_join1 = df_join1.withColumn(
    "time_diff_dep",
    spark_abs(
        unix_timestamp(col("DEP_DATETIME")) -
        unix_timestamp(col("OBS_DEP_DATETIME"))
    )
)

# Pour chaque vol, garder l'observation la plus proche
# par heure (fenêtre de 1h = 3600 secondes)
window_dep = Window.partitionBy(
    "FL_DATE",
    "ORIGIN_AIRPORT_ID",
    "DEST_AIRPORT_ID",
    "DEP_DATETIME"
).orderBy("time_diff_dep")

df_join1_closest = df_join1.withColumn(
    "rank_dep", row_number().over(window_dep)
).filter(col("rank_dep") == 1).drop("rank_dep", "time_diff_dep")

print(f"✅ Après sélection obs. plus proche : {df_join1_closest.count():,} lignes")
df_join1_closest.show(3, truncate=False)

# COMMAND ----------
# ================================
# CELLULE 4 — DEUXIÈME JOIN
# Section 4.1 du papier :
# W_d = météo destination de t_sa → t_sa-12h
# Clé join : (DEST_AIRPORT_ID, DATE(ARR))
# ================================

# Préparer OT pour jointure destination
df_OT_dest = df_OT.select(
    col("AirportID").alias("DEST_ID"),
    col("OBS_DATETIME").alias("OBS_ARR_DATETIME"),
    col("Temp").alias("dest_Temp"),
    col("Humidity").alias("dest_Humidity"),
    col("WindDirection").alias("dest_WindDir"),
    col("WindSpeed").alias("dest_WindSpeed"),
    col("Pressure").alias("dest_Pressure"),
    col("SkyCondition").alias("dest_SkyCondition"),
    col("Visibility").alias("dest_Visibility"),
    col("WeatherType").alias("dest_WeatherType")
)

# Deuxième Join :
# Pour chaque vol, trouver observations météo à destination
# dans la fenêtre [ARR_DATETIME - 12h, ARR_DATETIME]
df_join2 = df_join1_closest.join(
    df_OT_dest,
    (df_join1_closest["DEST_AIRPORT_ID"] == df_OT_dest["DEST_ID"]) &
    (df_OT_dest["OBS_ARR_DATETIME"] >=
     df_join1_closest["ARR_DATETIME"] - expr("INTERVAL 12 HOURS")) &
    (df_OT_dest["OBS_ARR_DATETIME"] <= df_join1_closest["ARR_DATETIME"]),
    "left"
)

print(f"✅ Après Deuxième Join : {df_join2.count():,} lignes")

# COMMAND ----------
# ================================
# CELLULE 5 — Garder observation
# météo la plus proche de t_sa
# ================================
df_join2 = df_join2.withColumn(
    "time_diff_arr",
    spark_abs(
        unix_timestamp(col("ARR_DATETIME")) -
        unix_timestamp(col("OBS_ARR_DATETIME"))
    )
)

window_arr = Window.partitionBy(
    "FL_DATE",
    "ORIGIN_AIRPORT_ID",
    "DEST_AIRPORT_ID",
    "DEP_DATETIME"
).orderBy("time_diff_arr")

df_JT = df_join2.withColumn(
    "rank_arr", row_number().over(window_arr)
).filter(col("rank_arr") == 1).drop("rank_arr", "time_diff_arr")

print(f"✅ Joint Table JT : {df_JT.count():,} lignes")

# COMMAND ----------
# ================================
# CELLULE 6 — Créer colonne classe C
# Section 4.2 du papier :
# C = 0 si ARR_DELAY < Th (on-time)
# C = 1 si ARR_DELAY >= Th (delayed)
# Deux seuils : 15min et 60min
# ================================
from pyspark.sql.functions import when

# Seuil 15 minutes
df_JT = df_JT.withColumn(
    "label_15",
    when(col("ARR_DELAY_NEW") >= 15, 1).otherwise(0)
)

# Seuil 60 minutes
df_JT = df_JT.withColumn(
    "label_60",
    when(col("ARR_DELAY_NEW") >= 60, 1).otherwise(0)
)

# Stats distribution classes
print("=== Distribution classes (threshold=15min) ===")
df_JT.groupBy("label_15").count().show()
# Attendu : ~80% ontime (0), ~20% delayed (1)

print("=== Distribution classes (threshold=60min) ===")
df_JT.groupBy("label_60").count().show()

# COMMAND ----------
# ================================
# CELLULE 7 — Vérification finale JT
# ================================
print("=== Colonnes JT ===")
print(df_JT.columns)

print("\n=== Aperçu JT ===")
df_JT.select(
    "FL_DATE",
    "ORIGIN_AIRPORT_ID",
    "DEST_AIRPORT_ID",
    "DEP_DATETIME",
    "ARR_DELAY_NEW",
    "orig_Temp", "orig_WindSpeed",
    "dest_Temp", "dest_WindSpeed",
    "label_15", "label_60"
).show(5, truncate=False)

# Vérifier pas de vols sans météo
n_sans_meteo_orig = df_JT.filter(
    col("orig_Temp").isNull()
).count()
n_sans_meteo_dest = df_JT.filter(
    col("dest_Temp").isNull()
).count()

print(f"⚠️  Vols sans météo origine      : {n_sans_meteo_orig:,}")
print(f"⚠️  Vols sans météo destination  : {n_sans_meteo_dest:,}")

# COMMAND ----------
# ================================
# CELLULE 8 — Sauvegarde JT
# ================================
df_JT.write.mode("overwrite").parquet(
    "/Volumes/workspace/default/outputs/JT/"
)

print("=" * 45)
print("     RÉSUMÉ FINAL — NOTEBOOK 02")
print("=" * 45)
print(f"✈️  Joint Table JT : {df_JT.count():,} vols")
print(f"📋 Colonnes       : {len(df_JT.columns)}")
print(f"🏷️  Label 15min   : {df_JT.filter(col('label_15')==1).count():,} delayed")
print(f"🏷️  Label 60min   : {df_JT.filter(col('label_60')==1).count():,} delayed")
print("=" * 45)
print("✅ Notebook 02 TERMINÉ → Prêt pour Notebook 03 !")